# Лабораторная работа 3

Тема: собственная MLP-сеть для бинарной классификации на датасетах d1/d2. Обучение выполняется нашими оптимизаторами через C++ flat-parameter MLP.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

import optlib

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASETS = {
    "d1": ROOT / "data" / "first_dataset.csv",
    "d2": ROOT / "data" / "second_dataset.csv",
}
print(optlib.__version__)

## Данные

CSV содержит признаки и последний столбец `target`. Разбиение 80/20 стратифицированное, стандартизация обучается только на train.

In [ ]:
for name, path in DATASETS.items():
    features, targets = optlib.load_csv(path)
    print(name, features.shape, np.bincount(targets.astype(int)))

## Обучение двумя оптимизаторами

Сравниваем Adam и HeavyBall. Оба метода используют один и тот же C++ MLP и Binary Cross-Entropy.

In [ ]:
rows = []
trained = {}
for dataset_name, path in DATASETS.items():
    for method, learning_rate in [("adam", 0.03), ("heavy_ball", 0.02)]:
        model, split, metrics = optlib.train_binary_classifier(
            path,
            method=method,
            hidden_dim=12,
            learning_rate=learning_rate,
            max_iter=3000,
            seed=42,
        )
        trained[(dataset_name, method)] = (model, split)
        rows.append({"dataset": dataset_name, "method": method, **metrics})

if pd is None:
    for row in rows:
        print(row)
else:
    display(pd.DataFrame(rows).drop(columns=["confusion_matrix"]))

## Confusion matrix

Матрицы ошибок показывают, что запас по F1 выше минимального порога 0.55.

In [ ]:
for row in rows:
    print(row["dataset"], row["method"])
    print(row["confusion_matrix"])

## Live-прогон d3

На защите закрытый набор можно скачать по Google Drive id и прогнать тем же pipeline. Если файл имеет тот же target-last формат, используется `evaluate(model, path, standardizer)`.

In [ ]:
# Пример для защиты:
# optlib.download("<google-drive-id>", ROOT / "data" / "third_dataset.csv")
# model, split = trained[("d1", "adam")]
# optlib.evaluate(model, ROOT / "data" / "third_dataset.csv", split.standardizer)
print("d1 id", optlib.FIRST_DATASET_ID)
print("d2 id", optlib.SECOND_DATASET_ID)